## Transformación


Una vez obtenidos los datos, es necesario transformarlos para que sean
consistentes y estén listos para su análisis. Esto incluye:

- Limpieza de datos: Eliminar registros duplicados, corregir errores
  tipográficos y manejar valores faltantes.
- Normalización: Asegurar que los datos estén en un formato uniforme, como
  convertir todas las fechas a un formato estándar.


In [1]:
import pandas as pd
from IPython.display import Markdown

from registro_de_indicencias.folder import CLEAN_CSV_PATH, CSV_PATH

In [2]:
df = pd.read_csv(CSV_PATH, sep=";", low_memory=False)

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 219373 entries, 0 to 219372
Data columns (total 18 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   N° CASO                219373 non-null  int64  
 1   ORIGEN                 219373 non-null  str    
 2   ZONA                   219228 non-null  str    
 3   BASE DESCENTRALIZADA   219373 non-null  str    
 4   TURNO                  219373 non-null  str    
 5   TIPO DE CASO           219373 non-null  str    
 6   FECHA DEL CASO         219373 non-null  int64  
 7   HORA DEL CASO          219373 non-null  str    
 8   ATENCION AL CASO       219373 non-null  int64  
 9   HORA ATENCION AL CASO  219373 non-null  int64  
 10  LATITUD                219373 non-null  float64
 11  LONGITUD               219373 non-null  float64
 12  ESTADO                 219373 non-null  str    
 13  FECHA CORTE            219373 non-null  int64  
 14  DEPARTAMENTO           219373 non-null  str    

Observamos que hay 17 columnas de las cuales, solo 1 es la que contiene valores
nulos, se procede a hacer una análisis en busca de tomar valores únicos.


In [3]:
# Current
count_rows = df.shape[0]
count_zona_na = df["ZONA"].isna().sum()
percentage_zona_na = (count_zona_na / count_rows) * 100

# Drop
df = df.dropna(subset=["ZONA"])
new_count_rows = df.shape[0]

display(
    Markdown(
        f"""\
Dado que las valores nulos en la columna **ZONA** representan el
${percentage_zona_na:.2f}\\%$ del total de filas, se decide eliminar las filas
con valores nulos en esta columna para asegurar la integridad de los datos en el
análisis posterior, dejando un total de ${new_count_rows}$ filas."""
    )
)

Dado que las valores nulos en la columna **ZONA** representan el
$0.07\%$ del total de filas, se decide eliminar las filas
con valores nulos en esta columna para asegurar la integridad de los datos en el
análisis posterior, dejando un total de $219228$ filas.

Ahora, procedemos a verificar todos las columnas que debería contener valores
únicos.


In [4]:
must_be_unique_columns = ["N° CASO"]
duplicated_columns = []


for column in must_be_unique_columns:
    text = []

    unique_values_count = df[column].nunique()
    has_unique_values = unique_values_count == len(df)

    if has_unique_values:
        text.append(f"- La columna **{column}** contiene valores únicos.")

        continue

    # Process the duplicated values
    text.append(f"- La columna **{column}** no contiene valores únicos.")

    # Show the unique values vs the total number of rows
    text.append(f"  - Valores únicos: ${unique_values_count}$")
    text.append(f"  - Total de filas: ${len(df)}$")
    text.append(f"  - Valores duplicados: ${len(df) - unique_values_count}$")
    text.append(
        f"  - Porcentaje de valores únicos: ${(unique_values_count / len(df)) * 100:.3f}\\%$"
    )

    # Display the duplicated rows for the column
    duplicated_rows = df[df.duplicated(subset=[column], keep=False)]
    text.append(
        f"\n{duplicated_rows[[*must_be_unique_columns, 'ORIGEN', 'ZONA', 'FECHA DEL CASO', 'HORA DEL CASO']].to_markdown(index=False)}"
    )

    duplicated_columns.append(column)

    display(Markdown("\n".join(text)))

df = df.drop_duplicates(subset=duplicated_columns)

- La columna **N° CASO** no contiene valores únicos.
  - Valores únicos: $219224$
  - Total de filas: $219228$
  - Valores duplicados: $4$
  - Porcentaje de valores únicos: $99.998\%$

|   N° CASO | ORIGEN   | ZONA                                 |   FECHA DEL CASO | HORA DEL CASO   |
|----------:|:---------|:-------------------------------------|-----------------:|:----------------|
|    801251 | CAMARA   | ZONA 1 / SECTOR 6 CIA DULANTO        |         10102025 | 10:35           |
|    801251 | CAMARA   | ZONA 1 / SECTOR 4 CIA RAMON CASTILLA |         10102025 | 10:35           |
|    856869 | CAMARA   | ZONA 1 / SECTOR 4 CIA RAMON CASTILLA |         29102025 | 17:42           |
|    856869 | CAMARA   | ZONA 3 / SECTOR 11 CIA OQUENDO       |         29102025 | 17:42           |
|    876165 | CAMARA   | ZONA 3 / SECTOR 11 CIA OQUENDO       |          4112025 | 1732            |
|    876165 | CAMARA   | ZONA 2 / SECTOR 9 CIA INGUNZA        |          4112025 | 1732            |
|    890838 | CAMARA   | ZONA 1 / SECTOR 4 CIA RAMON CASTILLA |          8112025 | 2107            |
|    890838 | CAMARA   | ZONA 1 / SECTOR 5 LA LEGUA           |          8112025 | 2107            |

Debido a que se tratan de casos casos donde se reportaron por más de una cámara,
se procede a eliminar debido a que al ser pocos no brindarían información
relevante al análisis. Teniendo en cuenta eso, se procede a verificar que las
columnas que debería tener un rango de valores como `ZONA`, no contengan valores
incongruentes como `NA` o `N/A`, o tener texto con minúscula o mayúscula (en
este caso se pasará todo a mayúscula).


In [5]:
uniform_columns = [
    "ORIGEN",
    "ZONA",
    "BASE DESCENTRALIZADA",
    "TURNO",
    "TIPO DE CASO",
    "ESTADO",
    "DEPARTAMENTO",
    "PROVINCIA",
    "DISTRITO",
    "UBIGEO",
]

for column in uniform_columns:
    text = []

    df[column] = df[column].astype(str).str.upper().str.strip()

    # Show the unique values for the column and the count
    unique_values = df[column].value_counts()

    text.append(f"- Columna **{column}** tiene los siguientes valores únicos:")
    for value, count in unique_values.items():
        text.append(f"  - `{value}`: ${count}$")

    display(Markdown("\n".join(text)))

- Columna **ORIGEN** tiene los siguientes valores únicos:
  - `CAMARA`: $213667$
  - `WHATSAPP`: $3498$
  - `APP CALLAO SEGURO`: $558$
  - `TELEFONO`: $475$
  - `BOTON DE PANICO`: $401$
  - `ATENCION PRESENCIAL`: $328$
  - `RADIO`: $238$
  - `QR - CONTACTO CIUDADANO`: $59$

- Columna **ZONA** tiene los siguientes valores únicos:
  - `ZONA 3 / SECTOR 11 CIA OQUENDO`: $46528$
  - `ZONA 2 / SECTOR 7 PLAYA RIMAC`: $35667$
  - `ZONA 2 / SECTOR 9 CIA INGUNZA`: $26744$
  - `ZONA 1 / SECTOR 1-2 CIA CALLAO`: $22121$
  - `ZONA 2 / SECTOR 8 BOCANEGRA`: $21826$
  - `ZONA 1 / SECTOR 4 CIA RAMON CASTILLA`: $17390$
  - `ZONA 1 / SECTOR 3 CIA LA CHALACA`: $15232$
  - `ZONA 1 / SECTOR 5 LA LEGUA`: $14304$
  - `ZONA 1 / SECTOR 6 CIA DULANTO`: $8239$
  - `ZONA 2 / SECTOR 10 CIA SARITA COLONIA`: $6468$
  - `ZONA 3 / SECTOR 12 CIA MARQUEZ`: $4701$
  - `ZONA 3 / SECTOR 12 CIA MARQUEZ,CLASE519ZONA 3 / SECTOR 11 CIA OQUENDO`: $2$
  - `ZONA 1 / SECTOR 1-2 CIA CALLAO,CLASE518ZONA 2 / SECTOR 9 CIA INGUNZA`: $1$
  - `ZONA 2 / SECTOR 9 CIA INGUNZA,CLASE517ZONA 2 / SECTOR 8 BOCANEGRA`: $1$

- Columna **BASE DESCENTRALIZADA** tiene los siguientes valores únicos:
  - `C - OQUENDO`: $49843$
  - `C - RAMON CASTILLA`: $32907$
  - `C - QUILCA`: $26352$
  - `C - TOMAS VALLE`: $22991$
  - `C - OBELISCO`: $18271$
  - `C2QUILCA`: $14218$
  - `BASE CENTRAL DE MONITOREO`: $13177$
  - `C2-BOCANEGRA`: $9683$
  - `C2 - RAMÓN CASTILLA`: $8633$
  - `C2 - OBELISCO`: $7159$
  - `C2 - TOMAS VALLE`: $6820$
  - `C - BOCANEGRA`: $5246$
  - `BD OVALO DE LAS AMERICAS`: $886$
  - `BD PARQUE LOS POZOS`: $677$
  - `BD PLAZA DE ARMAS MARQUEZ`: $550$
  - `BD MUJICA GALLO`: $330$
  - `BD PACASMAYO`: $212$
  - `BD SANTA MARINA SUR`: $200$
  - `BASE OVALO DE LAS AMERICAS`: $196$
  - `BD PARQUE DEL EJERCITO`: $168$
  - `BD ELMER FAUCETT`: $162$
  - `BD ALEJANDRO BERTELLO`: $154$
  - `BD ALFREDO PALACIOS`: $93$
  - `BD NUEVO AEROPUERTO`: $93$
  - `BD LOS DOMINICOS`: $66$
  - `BASE COSTANERA`: $40$
  - `BD OVALO CENTENARIO`: $30$
  - `BD SANTA ROSA`: $21$
  - `BD PLAZA SANTA ROSA`: $12$
  - `BD PARQUE 10 DE JUNIO`: $7$
  - `BASE BERTELLO`: $7$
  - `BASE SIN FRONTERAS`: $6$
  - `BASE PLAZA MARQUEZ`: $3$
  - `BASE TRANSPORTE`: $2$
  - `BASE DOMINICOS`: $2$
  - `BASE MANUEL MUJICA`: $2$
  - `BASE PARQUE DE JUNIO`: $2$
  - `BASE PARQUE LOS POZOS`: $1$
  - `BASE PLAZA SANTA ROSA`: $1$
  - `BASE PACASMAYO`: $1$

- Columna **TURNO** tiene los siguientes valores únicos:
  - `NOCHE`: $60041$
  - `MADRUGADA`: $58772$
  - `TARDE`: $54154$
  - `MAÑANA`: $46257$

- Columna **TIPO DE CASO** tiene los siguientes valores únicos:
  - `TRANSITO Y SEGURIDAD VIAL`: $141290$
  - `AMBIENTALES`: $36960$
  - `FISCALIZACION Y DEFENSA CIVIL`: $21842$
  - `APOYO AL CIUDADANO`: $16720$
  - `SEGURIDAD CIUDADANA`: $1771$
  - `APP CALLAO SEGURO`: $537$
  - `SALUD`: $37$
  - `PROTECCIÓN FAMILIAR`: $34$
  - `CASOS ESPECIALES`: $29$
  - `CIUDADANO`: $4$

- Columna **ESTADO** tiene los siguientes valores únicos:
  - `CERRADO`: $219224$

- Columna **DEPARTAMENTO** tiene los siguientes valores únicos:
  - `CALLAO`: $219224$

- Columna **PROVINCIA** tiene los siguientes valores únicos:
  - `CALLAO`: $219224$

- Columna **DISTRITO** tiene los siguientes valores únicos:
  - `CALLAO`: $219224$

- Columna **UBIGEO** tiene los siguientes valores únicos:
  - `70101`: $219224$

Teniendo en cuenta el contexto de los datos, confirmamos que los registros
provienen efectivamente del departamento, provincia y distrito del Callao, así
que procedemos a eliminar las columnas de `DEPARTAMENTO`, `PROVINCIA`,
`DISTRITO` para evitar redundancia en el análisis. Además, se eliminará la
columna `FECHA CORTE`, ya que contiene la fecha en la que se realizó el corte de
datos, lo cual no es relevante para el análisis de las incidencias. Del mismo,
la columna de `ESTADO` se eliminará debido a que todos los registros tienen el
mismo valor (`CERRADO`), lo cual no aporta información relevante para el
análisis. Otro aspecto a tener en cuenta es que existen valores incongruentes en
la columna `ZONA`, como
`ZONA 3 / SECTOR 12 CIA MARQUEZ,CLASE519ZONA 3 / SECTOR 11 CIA OQUENDO`, lo cual
no es correcto, por lo que se procederá a eliminar estos registros debido a que
son pocos y no aportan información relevante para el análisis.


In [6]:
df = df.drop(
    columns=[
        "DEPARTAMENTO",
        "PROVINCIA",
        "DISTRITO",
        "FECHA CORTE",
        "ESTADO",
        "UBIGEO",
    ]
)

In [7]:
invalid_values = [
    "ZONA 3 / SECTOR 12 CIA MARQUEZ,CLASE519ZONA 3 / SECTOR 11 CIA OQUENDO",
    "ZONA 1 / SECTOR 1-2 CIA CALLAO,CLASE518ZONA 2 / SECTOR 9 CIA INGUNZA",
    "ZONA 2 / SECTOR 9 CIA INGUNZA,CLASE517ZONA 2 / SECTOR 8 BOCANEGRA",
]

count_pre_drop_rows = df.shape[0]

df = df[~df["ZONA"].str.contains("|".join(invalid_values), regex=True)]

count_post_drop_rows = df.shape[0]

Markdown(
    f"""\
Se eliminaron las filas con valores inválidos en la columna **ZONA**, resultando
en la eliminación de ${count_pre_drop_rows - count_post_drop_rows}$ filas,
dejando un total de ${count_post_drop_rows}$ filas para el análisis posterior."""
)

Se eliminaron las filas con valores inválidos en la columna **ZONA**, resultando
en la eliminación de $4$ filas,
dejando un total de $219220$ filas para el análisis posterior.

Ahora procederemos a hacer la transformación de los datos de fecha y hora a su
formato correcto, para esto se utilizará la función `pd.to_datetime` de la
librería `pandas` para convertir las columnas `FECHA DEL CASO` y
`ATENCION AL CASO` a su formato de fecha, y lo mismo para `HORA DEL CASO` y
`HORA ATENCION AL CASO` para su formato de hora.


In [8]:
df["FECHA DEL CASO"] = pd.to_datetime(
    df["FECHA DEL CASO"].astype(str).str.zfill(8),
    errors="coerce",
    format="%d%m%Y",
)
df["HORA DEL CASO"] = pd.to_datetime(
    df["HORA DEL CASO"].str.replace(":", "").str.zfill(4),
    errors="coerce",
    format="%H%M",
).dt.time
df["ATENCION AL CASO"] = pd.to_datetime(
    df["ATENCION AL CASO"].astype(str).str.zfill(8),
    errors="coerce",
    format="%d%m%Y",
)
df["HORA ATENCION AL CASO"] = pd.to_datetime(
    df["HORA ATENCION AL CASO"].astype(str).str.zfill(4),
    errors="coerce",
    format="%H%M",
).dt.time

Finalmente, se muestra un resumen de las columnas con su tipo de dato, cantidad
de valores nulos y no nulos, para verificar que la transformación se haya
realizado correctamente. El resultado final se exportará a un nuevo archivo CSV
para el proceso de carga.


In [9]:
df.to_csv(CLEAN_CSV_PATH, index=False)


resumen = pd.DataFrame({
    "Columna": df.columns,
    "Tipo": df.dtypes.array,
    "No nulos": df.notna().sum().array,
    "Nulos": df.isna().sum().array,
})

Markdown(resumen.to_markdown(index=False))

| Columna               | Tipo           |   No nulos |   Nulos |
|:----------------------|:---------------|-----------:|--------:|
| N° CASO               | int64          |     219220 |       0 |
| ORIGEN                | str            |     219220 |       0 |
| ZONA                  | str            |     219220 |       0 |
| BASE DESCENTRALIZADA  | str            |     219220 |       0 |
| TURNO                 | str            |     219220 |       0 |
| TIPO DE CASO          | str            |     219220 |       0 |
| FECHA DEL CASO        | datetime64[us] |     219220 |       0 |
| HORA DEL CASO         | object         |     219220 |       0 |
| ATENCION AL CASO      | datetime64[us] |     219220 |       0 |
| HORA ATENCION AL CASO | object         |     219220 |       0 |
| LATITUD               | float64        |     219220 |       0 |
| LONGITUD              | float64        |     219220 |       0 |